# Decay files

This notebook runs entirely in your browser through [Pyodide](https://pyodide.org). The first cell installs `pyevtgen` from its package index.

In [ ]:
import micropip

await micropip.install(
    "pyevtgen", index_urls="https://redeboer.github.io/pyevtgen-test/simple"
)

A user decay file overrides entries of the default decay table. Edit the model in the file below, for instance `D_DALITZ` or `PHSP`, and rerun the cells to compare.

Creating a `Generator` reads the decay table and initializes Pythia8. In the browser, this takes about 15 seconds and prints a lot of log output. A notebook can create only one `Generator`, so restart the kernel to create another one.

In [ ]:
from pathlib import Path

from pyevtgen import Generator, final_state_momenta

generator = Generator(seed=0)

In [ ]:
decay_file = Path("dplus.dec")
decay_file.write_text("""
Decay D+
1.0 K- pi+ pi+ D_DALITZ;
Enddecay
End
""")
generator.read_user_decay_file(decay_file)
momenta = final_state_momenta(generator.generate("D+", n=20_000))
momenta.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def mass_squared(p: np.ndarray) -> np.ndarray:
    return p[..., 0] ** 2 - np.sum(p[..., 1:] ** 2, axis=-1)


fig, ax = plt.subplots(figsize=(4.5, 4))
ax.hist2d(
    mass_squared(momenta[:, 0] + momenta[:, 1]),
    mass_squared(momenta[:, 0] + momenta[:, 2]),
    bins=60,
    cmin=1,
)
ax.set_xlabel(r"$m^2(K^-\pi^+_1)$ [GeV$^2$]")
ax.set_ylabel(r"$m^2(K^-\pi^+_2)$ [GeV$^2$]")
plt.show()